# Lab 2 — Worked solutions

Solutions to [Lab_DataEngineering.ipynb](./Lab_DataEngineering.ipynb). There
are many correct variants of each answer; these are the most idiomatic.

## Exercise 1: load the data

In [1]:
import pandas as pd

df = pd.read_csv('breast_cancer_data.csv', index_col='id', header=0)
print(df.shape)
print(df.columns.tolist())

(569, 31)
['diagnosis', 'Radius_mean', 'Texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']


`index_col='id'` makes the sample identifier the index of the DataFrame, and
`header=0` (the default) takes row 0 as the column names.

## Exercise 2: normalise the column names

In [2]:
print([c for c in df.columns if c != c.lower()])

df = df.rename(columns={'Radius_mean': 'radius_mean', 'Texture_mean': 'texture_mean'})
print([c for c in df.columns if c != c.lower()])   # nothing left []

['Radius_mean', 'Texture_mean']
[]


## Exercise 3: query a subset

In [3]:
low_concavity = df.loc[df['concavity_mean'] < 0.1]
print(len(low_concavity))
low_concavity.head()

373


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
id,,,,,,,,,,,,,,,,,,,,,
842517,M,20.57,21.77,132.9,1326,0.08474,0.07864,0.08690,0.07017,0.1812,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.18600,0.2750,0.08902
84458202,M,13.71,20.83,90.2,577.9,0.11890,0.16450,0.09366,0.05985,0.2196,...,17.06,28.14,110.6,897.0,0.1654,0.3682,0.2678,0.15560,0.3196,0.11510
845636,M,16.02,23.24,102.7,797.8,0.08206,0.06669,0.03299,0.03323,0.1528,...,19.19,33.88,123.8,1150.0,0.1181,0.1551,0.1459,0.09975,0.2948,0.08452
84610002,M,15.78,17.89,103.6,781,0.09710,0.12920,0.09954,0.06606,0.1842,...,20.42,27.28,136.5,1299.0,0.1396,0.5609,0.3965,0.18100,0.3792,0.10480
846381,M,15.85,23.95,103.7,782.7,0.08401,0.10020,0.09938,0.05364,0.1847,...,16.84,27.66,112.0,876.5,0.1131,0.1924,0.2322,0.11190,0.2809,0.06287


## Exercise 4: descriptive statistics

In [4]:
p = df['perimeter_mean']
print(f"mean {p.mean():.3f}, std {p.std():.3f}, var {p.var():.3f}")

# Equivalently, one line:
df['perimeter_mean'].agg(['mean', 'std', 'var'])

mean 91.969, std 24.299, var 590.440


mean     91.969033
std      24.298981
var     590.440480
Name: perimeter_mean, dtype: float64

## Exercise 5: class balance

In [5]:
pct_malignant = 100 * (df['diagnosis'] == 'M').mean()
print(f"{pct_malignant:.2f}% malignant")

37.26% malignant


The trick: `df['diagnosis'] == 'M'` is a boolean Series, and booleans behave
as 0/1 under arithmetic, so `.mean()` is exactly the fraction of `True`s.

## Exercise 6: corrupted values to `NaN`

In [6]:
print(df.dtypes[df.dtypes == object])   # 'diagnosis' is rightly text; the rest is corruption

feature_cols = [c for c in df.columns if c != 'diagnosis']
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df.isnull().sum()[lambda s: s > 0])
print("rows containing at least one NaN:", df.index[df.isnull().any(axis=1)].tolist())

diagnosis            object
area_mean            object
symmetry_mean        object
concave points_se    object
symmetry_se          object
dtype: object
texture_mean               1
area_mean                  1
smoothness_mean            1
symmetry_mean              1
fractal_dimension_mean     2
area_se                    1
compactness_se             1
concave points_se          1
symmetry_se                1
fractal_dimension_se       1
texture_worst              1
concave points_worst       1
fractal_dimension_worst    1
dtype: int64
rows containing at least one NaN: [8911800, 89122, 8912284, 891670, 891716, 90251, 90291, 903516, 907915, 908489, 911296202, 9113239, 917092, 91813701]


`errors='coerce'` turns anything that does not parse as a number — the junk
token `'A'` and the empty cells alike — into `NaN`, so every corrupted and
empty entry becomes a proper missing value. Note the `diagnosis` column is
explicitly excluded: filling it with a number would destroy the labels we
need for classification.

## Exercise 7: impute the missing values

In [7]:
for col in feature_cols:
    df[col] = df[col].fillna(df[col].median())

print("NaN left:", df.isnull().sum().sum())

NaN left: 0


**Why the median?** The mean is dragged towards extreme values, so after
imputation the filled values inherit any outlier problem (see the lecture's
"order matters" example). The median is *robust*: up to half the data can be
extreme before the median moves. We impute only the numeric feature columns
so `diagnosis` survives intact.

## Exercise 8: zeros aren't real — remove and save

In [8]:
zero_mask = (df[feature_cols] == 0).any(axis=1)
print("rows to drop:", zero_mask.sum())

df_clean = df.loc[~zero_mask]
print(df_clean.shape)

df_clean.to_csv('breast_cancer_data_clean.csv', index_label='id')

rows to drop: 13
(556, 31)


`(df[feature_cols] == 0)` is a boolean DataFrame; `.any(axis=1)` collapses it
to "does this row contain any zero", and `~` negates the mask so `loc` keeps
the clean rows. The final dataset has 556 rows × 31 columns.